[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/06-evaluare-optimizare/extra-06-exemplu-clasificator-svm.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# În Colab descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    %cd /content/SEED/cursuri/06-evaluare-optimizare
sys.path.insert(0, os.path.abspath("../.."))

In [1]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn import metrics


In [2]:
df_heart = pd.read_csv('./date/Heart.csv',header = 0).iloc[:,1:]
df_heart = df_heart.dropna(how = 'any')
df_heart['AHD']= df_heart["AHD"].apply(lambda x: 1 if x == 'Yes' else 0)
df_heart = pd.get_dummies(df_heart, columns = ['ChestPain','Thal'])
df_heart.info()

<class 'pandas.core.frame.DataFrame'>
Index: 297 entries, 0 to 301
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Age                     297 non-null    int64  
 1   Sex                     297 non-null    int64  
 2   RestBP                  297 non-null    int64  
 3   Chol                    297 non-null    int64  
 4   Fbs                     297 non-null    int64  
 5   RestECG                 297 non-null    int64  
 6   MaxHR                   297 non-null    int64  
 7   ExAng                   297 non-null    int64  
 8   Oldpeak                 297 non-null    float64
 9   Slope                   297 non-null    int64  
 10  Ca                      297 non-null    float64
 11  AHD                     297 non-null    int64  
 12  ChestPain_asymptomatic  297 non-null    bool   
 13  ChestPain_nonanginal    297 non-null    bool   
 14  ChestPain_nontypical    297 non-null    bool   

In [3]:
X = df_heart[[c for c in df_heart.columns if c != 'AHD']]
y = df_heart['AHD']

## Exemplu de antrenare clasificator SVM

In [4]:
from sklearn.model_selection import GridSearchCV


def antrenament_scikitlearn_svmclsf(X,y):
    
    # instațiem un model
    svmclsf = svm.SVC()#random_state=42)
    
    # setăm parametrii pe care vrem să îi încercăm
    param_grid ={'kernel':['linear','poly','rbf'],
             'C':[ 0.5 ,0.1, 1, 10, 100, 1000]}
    
    # folosim  antrenare cu validare încrucișată
    CV_svmclsf = GridSearchCV(
                        estimator=svmclsf, 
                          param_grid=param_grid, 
                          scoring = 'accuracy', # aici e diferenta fata de clasificator - metrica aleasa pentru evaluare
                          cv = 5, # validare incrucistata cu impartire in 5 seturi egale de date
                          n_jobs = 6, # nr de sarcini paralele execute
                         error_score='raise')
    # pasul de antrenare
    CV_svmclsf.fit(X, y)
    
    # returnăm cei mai buni parametrii și cel mai bun model
    return CV_svmclsf.best_params_, CV_svmclsf.best_estimator_

In [5]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

# 5 iteratii 
results_CV_paralel = []
for i in range(5):
    
    print("Iteration {0}".format(i))
    # Split dataset în set de antrenament și set de test 
    X_train, X_test, y_train, y_test = train_test_split(X, 
                                                        df_heart['AHD'], 
                                                        test_size=0.3,
                                                       stratify = df_heart['AHD'],
                                                       shuffle = True) # 70% training / 30% test

    params,model = antrenament_scikitlearn_svmclsf(X_train, y_train)
    
    # prognozăm cu cel mai bun model
    y_test_pred = model.predict(X_test)
    results_CV_paralel.append({
        'accuracy_score': accuracy_score(y_true = y_test, y_pred=y_test_pred),
        'parametrii': params}
        )
    
    
print("Acuratete medie:", sum([r['accuracy_score'] for r in results_CV_paralel])/len(results_CV_paralel))

pd.DataFrame(results_CV_paralel)

Iteration 0
Iteration 1
Iteration 2
Iteration 3
Iteration 4
Acuratete medie: 0.8111111111111111


,accuracy_score,parametrii
0,0.800000,"{'C': 0.1, 'kernel': 'linear'}"
1,0.833333,"{'C': 1, 'kernel': 'linear'}"
2,0.855556,"{'C': 0.1, 'kernel': 'linear'}"
3,0.755556,"{'C': 0.5, 'kernel': 'linear'}"
4,0.811111,"{'C': 1000, 'kernel': 'poly'}"


## Exemplu de antrenare regresor Random Classifier cu GridSearch

In [10]:
from sklearn.ensemble import RandomForestClassifier




In [ ]:
def antrenament_scikitlearn_rfc(X,y):
    
    # instațiem un model
    model = RandomForestClassifier()#random_state=42)
    
    # setăm parametrii pe care vrem să îi încercăm
    param_grid = {
                    'n_estimators': [100,1000], #[100, 200, 500, 1000],
                    'max_features': ['sqrt','log2', int(round(X.shape[0]/3,0)) ], #cutuma pt regresie: 1/3 din numarul de atribute
                    'max_samples' : [0.1, 0.2, 0.3, 0.4] ,
                    #'criterion' :['gini', 'entropy']
                 }
    
    # folosim  antrenare cu validare încrucișată
    CV_model = GridSearchCV(estimator=model, 
                          param_grid=param_grid, 
                          scoring = 'accuracy', # aici e diferenta fata de clasificator - metrica aleasa pentru evaluare
                          cv = 10, 
                          n_jobs = 6,
                         error_score='raise')
    # pasut de antrenare
    CV_model.fit(X, y)
    
    # returnăm cei mai buni parametrii și cel mai bun model
    return CV_model.best_params_, CV_model.best_estimator_



In [12]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

# 5 iteratii 
results_CV_paralel = []
for i in range(5):
    
    print("Iteration {0}".format(i))
    # Split dataset în set de antrenament și set de test 
    X_train, X_test, y_train, y_test = train_test_split(X, 
                                                        df_heart['AHD'], 
                                                        test_size=0.3,
                                                       stratify = df_heart['AHD'],
                                                       shuffle = True) # 70% training / 30% test

    params,model = antrenament_scikitlearn_rfc(X_train, y_train)
    
    # prognozăm cu cel mai bun model
    y_test_pred = model.predict(X_test)
    results_CV_paralel.append({
        'accuracy_score': accuracy_score(y_true = y_test, y_pred=y_test_pred),
        'parametrii': params}
        )
    
    
print("Acuratete medie:", sum([r['accuracy_score'] for r in results_CV_paralel])/len(results_CV_paralel))

pd.DataFrame(results_CV_paralel)

Iteration 0
Iteration 1
Iteration 2
Iteration 3
Iteration 4
Acuratete medie: 0.828888888888889


,accuracy_score,parametrii
0,0.788889,"{'max_features': 69, 'max_samples': 0.3, 'n_estimators': 1000}"
1,0.844444,"{'max_features': 69, 'max_samples': 0.1, 'n_estimators': 100}"
2,0.833333,"{'max_features': 'sqrt', 'max_samples': 0.1, 'n_estimators': 1000}"
3,0.833333,"{'max_features': 'sqrt', 'max_samples': 0.4, 'n_estimators': 1000}"
4,0.844444,"{'max_features': 'sqrt', 'max_samples': 0.2, 'n_estimators': 100}"


In [13]:
print("Average MAE:", sum([r['accuracy_score'] for r in results_CV_paralel])/len(results_CV_paralel))

pd.DataFrame(results_CV_paralel)

Average MAE: 0.828888888888889


,accuracy_score,parametrii
0,0.788889,"{'max_features': 69, 'max_samples': 0.3, 'n_estimators': 1000}"
1,0.844444,"{'max_features': 69, 'max_samples': 0.1, 'n_estimators': 100}"
2,0.833333,"{'max_features': 'sqrt', 'max_samples': 0.1, 'n_estimators': 1000}"
3,0.833333,"{'max_features': 'sqrt', 'max_samples': 0.4, 'n_estimators': 1000}"
4,0.844444,"{'max_features': 'sqrt', 'max_samples': 0.2, 'n_estimators': 100}"


# Exemplu de antrenare regresor XGBoost cu Grid Search pentru parametrii

In [14]:
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
def antrenament_scikitlearn_xgb_c(X,y):
    
    model = GradientBoostingClassifier()#random_state=42)
    param_grid = {
                    'n_estimators' : [10, 100, 1000],
                    'learning_rate' : [0.001, 0.01, 0.1],
                    'subsample' : [0.2, 0.6, 0.8],
                    'max_depth' : [3, 7, 9]
                 }
     # folosim  antrenare cu validare încrucișată
    CV_model = GridSearchCV(estimator=model, 
                          param_grid=param_grid, 
                          scoring = 'accuracy', # aici e diferenta fata de clasificator - metrica aleasa pentru evaluare
                          cv = 10, 
                          n_jobs = 6,
                         error_score='raise')
    # pasut de antrenare
    CV_model.fit(X, y)
    
    # returnăm cei mai buni parametrii și cel mai bun model
    return CV_model.best_params_, CV_model.best_estimator_



Apelarea functiei de antrenament:

In [16]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

# 5 iteratii 
results_CV_paralel = []
for i in range(5):
    
    print("Iteration {0}".format(i))
    # Split dataset în set de antrenament și set de test 
    X_train, X_test, y_train, y_test = train_test_split(X, 
                                                        df_heart['AHD'], 
                                                        test_size=0.3,
                                                       stratify = df_heart['AHD'],
                                                       shuffle = True) # 70% training / 30% test

    params,model = antrenament_scikitlearn_xgb_c(X_train, y_train)
    
    # prognozăm cu cel mai bun model
    y_test_pred = model.predict(X_test)
    results_CV_paralel.append({
        'accuracy_score': accuracy_score(y_true = y_test, y_pred=y_test_pred),
        'parametrii': params}
        )
    
    
print("Acuratete medie:", sum([r['accuracy_score'] for r in results_CV_paralel])/len(results_CV_paralel))

pd.DataFrame(results_CV_paralel)

Iteration 0
Iteration 1
Iteration 2
Iteration 3
Iteration 4
Acuratete medie: 0.8044444444444444


,accuracy_score,parametrii
0,0.822222,"{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.2}"
1,0.811111,"{'learning_rate': 0.001, 'max_depth': 3, 'n_estimators': 1000, 'subsample': 0.2}"
2,0.777778,"{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 1000, 'subsample': 0.8}"
3,0.788889,"{'learning_rate': 0.001, 'max_depth': 3, 'n_estimators': 1000, 'subsample': 0.2}"
4,0.822222,"{'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 100, 'subsample': 0.2}"


# COMPARARE CLASIFICATORI:
1. spargeti setul de date in X_train,X_test_y_train,y_test =train_test_split(X, 
                                                        df_proiect
                                                        test_size=0.3,
                                                       stratify = df_proiect[<nume variabila tinta>],
                                                       shuffle = True)
    
 2. antrenati 3 tipuri de clasificatori diferiti pe X_train
    
 3. evaluati clasficatorii pe acelasi set de date nefolosit la antrenament, X_test.
    
 

In [ ]:

# Split dataset în set de antrenament și set de test 
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    df_heart['AHD'], 
                                                    test_size=0.3,
                                                   stratify = df_heart['AHD'],
                                                   shuffle = True) # 70% training / 30% test

params,model = antrenament_scikitlearn_xgb_c(X_train, y_train)

# prognozăm cu cel mai bun model
y_test_pred = model.predict(X_test)
results_CV_paralel.append({
    'metoda':'xgb',
    'accuracy_score': accuracy_score(y_true = y_test, y_pred=y_test_pred),
    'parametrii': params}
    )
params,model = antrenament_scikitlearn_svmclsf (X_train, y_train)
# prognozăm cu cel mai bun model
y_test_pred = model.predict(X_test)
results_CV_paralel.append({
    'metoda':'svmcls',
    'accuracy_score': accuracy_score(y_true = y_test, y_pred=y_test_pred),
    'parametrii': params}
    )

params,model = antrenament_scikitlearn_rfc (X_train, y_train)
# prognozăm cu cel mai bun model
y_test_pred = model.predict(X_test)
results_CV_paralel.append({
    'metoda':'decision tree classifier',
    'accuracy_score': accuracy_score(y_true = y_test, y_pred=y_test_pred),
    'parametrii': params}
    )

#... random forest
#... logistic regression